In [1]:
import os
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
import json
import gzip

In [2]:
num_workers = 4  
fastq_dir = Path("./fastq")  
symlink_dir = Path("./symlinks") # Temporary clean-named links
#results_dir = Path("results")
snp_only_dir = Path("./snp_only")
logs_dir = Path("./logs")
tmp_dir = Path("./tmp")
vcf_dir = Path("./vcf")

In [3]:
#results_dir.mkdir(exist_ok=True)
fastq_dir.mkdir(exist_ok=True)
symlink_dir.mkdir(exist_ok=True)
snp_only_dir.mkdir(exist_ok=True)
logs_dir.mkdir(exist_ok=True)
tmp_dir.mkdir(exist_ok=True)
vcf_dir.mkdir(exist_ok=True)

In [ ]:
# Group FASTQs by ENA run ID
# samples = {}

# for fq in fastq_dir.glob("*.fastq.gz"):
#     name = fq.name

#     if fq.name.endswith(".aria2"):
#         #print(f"⚠️ Skipping {fq.name}: still downloading (.aria2).")
#         continue

#     # Match ERR / SRR with regex
#     match = re.search(r'([SE]RR\d+)', name)
#     if not match:
#         print(f"⚠️ Skipping {name}: no ENA run ID found.")
#         continue

#     clean_base = match.group(1)

#     # Figure out _1 or _2
#     if "_1" in name:
#         suffix = "_1"
#     elif "_2" in name:
#         suffix = "_2"
#     else:
#         suffix = ""
#     # else:
#     #     print(f"⚠️ Skipping {name}: no read pair info.")
#     #     continue

#     # Make symlink with correct name
#     symlink_name = f"{clean_base}{suffix}.fastq.gz"
#     symlink_path = symlink_dir / symlink_name

#     if not symlink_path.exists():
#         symlink_path.symlink_to(fq.resolve())

#     samples.setdefault(clean_base, []).append(symlink_path)

# print(f"✅ Found {len(samples)} samples to process.")

In [4]:
samples = {}
country_map = {}

for fq in fastq_dir.glob("*.fastq.gz"):
    name = fq.name

    if fq.name.endswith(".aria2"):
        continue

    match = re.search(r'([SE]RR\d+)', name)
    if not match:
        print(f"⚠️ Skipping {name}: no ENA run ID found.")
        continue

    clean_base = match.group(1)

    parts = name.split("_")
    country = "_".join(parts[:-2])
    country_map[clean_base] = country

    #suffix = "_1" if "_1" in name else "_2" if "_2" in name else ""
    # Figure out _1 or _2
    if "_1" in name:
        suffix = "_1"
    elif "_2" in name:
        suffix = "_2"
    else:
        suffix = ""
    symlink_name = f"{clean_base}{suffix}.fastq.gz"
    symlink_path = symlink_dir / symlink_name

    if not symlink_path.exists():
        symlink_path.symlink_to(fq.resolve())

    samples.setdefault(clean_base, []).append(symlink_path)

print(f"✅ Found {len(samples)} samples to process.")

# Save mapping
with open("country_map.json", "w") as f:
    json.dump(country_map, f, indent=2)

✅ Found 580 samples to process.


In [ ]:
# Function to process one sample
# def process_sample(sample, files):
#     files = sorted(files)
#     output_prefix = sample

#     if len(files) == 2:
#         fq1, fq2 = files
#         tb_cmd = [
#             "tb-profiler",
#             "profile",
#             "-1", str(fq1),
#             "-2", str(fq2),
#             "-p", str(output_prefix),
#             "--txt",
#             "--temp", "./tmp"
#         ]
#     elif len(files) == 1:
#         fq1 = files[0]
#         tb_cmd = [
#             "tb-profiler",
#             "profile",
#             "-1", str(fq1),
#             "-p", str(output_prefix),
#             "--txt",
#             "--temp", "./tmp"
#         ]
#     else:
#         print(f"⚠️ Skipping {sample}: no valid FASTQ files.")
#         return

#     print(f"🔬 Running TB-Profiler for {sample}")
#     with open(logs_dir / f"{sample}_tbprofiler.log", "w") as log_file:
#         subprocess.run(tb_cmd, stdout=log_file, stderr=log_file, check=True)

#     input_vcf = vcf_dir / f"{output_prefix}.targets.vcf.gz"
#     output_vcf = snp_only_dir / f"{sample}_snps_only.vcf"

#     # STEP 1 — Extract SNPs only
#     bcf_cmd = [
#         "bcftools", "view",
#         "-v", "snps",
#         input_vcf,
#         "-o", str(output_vcf),
#         "--output-type", "v"
#     ]

#     print(f"🧬 Extracting SNPs for {sample}")
#     with open(logs_dir / f"{sample}_bcftools.log", "w") as log_file:
#         subprocess.run(bcf_cmd, stdout=log_file, stderr=log_file, check=True)

#     print(f"✅ Finished: {sample}")

#     # STEP 2 — Filter SNPs with DP, QUAL, AF thresholds
#     filtered_vcf = snp_only_dir / f"{sample}_snps_only_filtered.vcf"
#     bcf_filter_cmd = [
#         "bcftools", "filter",
#         "-i", "DP>=5 && QUAL>=20 && AF>=0.75",
#         str(output_vcf),
#         "-o", str(filtered_vcf)
#     ]

#     print(f"🧪 Filtering SNPs for {sample} with DP>=5, QUAL>=20, AF>=0.75")
#     with open(logs_dir / f"{sample}_bcftools_filter.log", "w") as log_file:
#         subprocess.run(bcf_filter_cmd, stdout=log_file, stderr=log_file, check=True)

#     print(f"✅ Finished: {sample} — Final filtered SNPs: {filtered_vcf}")

In [5]:
def is_gzip_ok(fq_file):
    try:
        with gzip.open(fq_file, 'rb') as f:
            f.read(1024)  # Just read first 1KB
        return True
    except OSError:
        return False

def process_sample(sample, files):
    files = sorted(files)
    output_prefix = sample

    # ✅ Check if files are valid gzips
    for fq in files:
        if not is_gzip_ok(fq):
            print(f"⚠️ Skipping {sample}: File {fq} is not a valid gzip FASTQ!")
            return

    if len(files) == 2:
        fq1, fq2 = files
        tb_cmd = [
            "tb-profiler",
            "profile",
            "-1", str(fq1),
            "-2", str(fq2),
            "-p", str(output_prefix),
            "--txt",
            "--temp", "./tmp"
        ]
    elif len(files) == 1:
        fq1 = files[0]
        tb_cmd = [
            "tb-profiler",
            "profile",
            "-1", str(fq1),
            "-p", str(output_prefix),
            "--txt",
            "--temp", "./tmp"
        ]
    else:
        print(f"⚠️ Skipping {sample}: no valid FASTQ files.")
        return

    print(f"🔬 Running TB-Profiler for {sample}")
    with open(logs_dir / f"{sample}_tbprofiler.log", "w") as log_file:
        subprocess.run(tb_cmd, stdout=log_file, stderr=log_file, check=True)

    input_vcf = vcf_dir / f"{output_prefix}.targets.vcf.gz"
    output_vcf = snp_only_dir / f"{sample}_snps_only.vcf"

    # STEP 1 — Extract SNPs only
    bcf_cmd = [
        "bcftools", "view",
        "-v", "snps",
        input_vcf,
        "-o", str(output_vcf),
        "--output-type", "v"
    ]

    print(f"🧬 Extracting SNPs for {sample}")
    with open(logs_dir / f"{sample}_bcftools.log", "w") as log_file:
        subprocess.run(bcf_cmd, stdout=log_file, stderr=log_file, check=True)

    print(f"✅ Finished: {sample}")

    # STEP 2 — Filter SNPs with DP, QUAL, AF thresholds
    filtered_vcf = snp_only_dir / f"{sample}_snps_only_filtered.vcf"
    bcf_filter_cmd = [
        "bcftools", "filter",
        "-i", "DP>=5 && QUAL>=20 && AF>=0.75",
        str(output_vcf),
        "-o", str(filtered_vcf)
    ]

    print(f"🧪 Filtering SNPs for {sample} with DP>=5, QUAL>=20, AF>=0.75")
    with open(logs_dir / f"{sample}_bcftools_filter.log", "w") as log_file:
        subprocess.run(bcf_filter_cmd, stdout=log_file, stderr=log_file, check=True)

    print(f"✅ Finished: {sample} — Final filtered SNPs: {filtered_vcf}")


In [ ]:
# Run with parallel jobs 
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = [executor.submit(process_sample, sample, files) for sample, files in samples.items()]
    for future in as_completed(futures):
        try:
            future.result()
        except subprocess.CalledProcessError as e:
            print(f"❌ Error running command: {e}")
print("🎉 All samples done.")

# Previous code was running in a loop, but now we use ThreadPoolExecutor to run all samples in parallel.
# with ThreadPoolExecutor(max_workers=num_workers) as executor:
#     futures = []
#     for sample, files in samples.items():
#         futures.append(executor.submit(process_sample, sample, files))

#     for future in as_completed(futures):
#         try:
#             future.result()
#         except subprocess.CalledProcessError as e:
#             print(f"❌ Error running command: {e}") # meron pa ring error like ERR4810720, ERR2199934, SRR6152742 figure out why

print("🎉 All samples done.")
# 8:34 pm - terminated as no new files are being downloaded
# 9:02 pm

# July 25: 10:14 pm
# July 25: 10:41 pm

🔬 Running TB-Profiler for SRR6339649
🔬 Running TB-Profiler for SRR6650306
🔬 Running TB-Profiler for ERR4830728
⚠️ Skipping SRR6650276: no valid FASTQ files.
⚠️ Skipping SRR6650422: no valid FASTQ files.
⚠️ Skipping SRR6650425: no valid FASTQ files.
⚠️ Skipping SRR6650354: no valid FASTQ files.
🔬 Running TB-Profiler for SRR6650224
🧬 Extracting SNPs for SRR6339649
❌ Error running command: Command '['bcftools', 'view', '-v', 'snps', PosixPath('vcf/SRR6339649.targets.vcf.gz'), '-o', 'snp_only/SRR6339649_snps_only.vcf', '--output-type', 'v']' returned non-zero exit status 255.
🔬 Running TB-Profiler for SRR6650353
🧬 Extracting SNPs for SRR6650353
❌ Error running command: Command '['bcftools', 'view', '-v', 'snps', PosixPath('vcf/SRR6650353.targets.vcf.gz'), '-o', 'snp_only/SRR6650353_snps_only.vcf', '--output-type', 'v']' returned non-zero exit status 255.
⚠️ Skipping SRR6650301: File symlinks/SRR6650301_2.fastq.gz is not a valid gzip FASTQ!
⚠️ Skipping SRR6650289: no valid FASTQ files.
⚠️ S